In [ ]:
import sys
import pandas as pd
from astropy.coordinates import SkyCoord, match_coordinates_sky
from astropy import units as u 

if './SelfCalGroupFinder/py/' not in sys.path:
    sys.path.append('./SelfCalGroupFinder/py/')
from pyutils import *
from dataloc import *
from bgs_helpers import *
import catalog_definitions as cat
from groupcatalog import *

%load_ext autoreload
%autoreload 2

In [ ]:
# BGS data to get the centrals we want to stack on
gc = deserialize(cat.bgs_y3_pzp_2_6_c2)
centrals_df = gc.centrals

In [ ]:
# Read the Lsat file into a DataFrame.
lsat_df = pd.read_csv(LSAT_RAW_2020_OBSERVATIONS_FILE)
print(lsat_df.columns)

# How many did Jeremy's original analysis find as centrals?
print(f"There are {len(lsat_df)} rows in the Lsat file, and {len(lsat_df.loc[lsat_df['p_sat'] < 0.5])} are centrals in the SDSS analysis.")

# Ensure no nan values anywhere in file
lsat_df = lsat_df.loc[:, ['RA', 'Dec', 'z', 'Dn4k', 'absmag_r', 'Ltot100', 'Ltot50', 'Rbar']]
orig_rows = len(lsat_df)
lsat_df = lsat_df.dropna()
print(f"Lsat data acquired for {len(lsat_df)} galaxies. Dropped {orig_rows - len(lsat_df)} rows with NaN values.")

In [ ]:
def get_lsat_for_centrals(lsat_file, centrals_df):
    """
    Get the Lsat data for central galaxies by matching on RA and DEC.
    """
    orig_bgs_len = len(centrals_df)

    # Read the Lsat file into a DataFrame.
    lsat_df = pd.read_csv(lsat_file)
    lsat_df = lsat_df.loc[:, ['RA', 'Dec', 'z', 'Dn4k', 'absmag_r', 'Ltot100', 'Ltot50', 'Rbar']]
    lsat_df = lsat_df.dropna() # No nans are present anyway
    print(f"Lsat data acquired for {len(lsat_df)} galaxies.")

    # Use match_coordinates_sky to match the the RA,DEC
    lsat_coord = SkyCoord(ra=lsat_df['RA'].to_numpy()*u.degree, dec=lsat_df['Dec'].to_numpy()*u.degree, frame='icrs')
    central_coord = SkyCoord(ra=centrals_df['RA'].to_numpy()*u.degree, dec=centrals_df['DEC'].to_numpy()*u.degree, frame='icrs')
    idx, d2d, d3d  = match_coordinates_sky(central_coord, lsat_coord, nthneighbor=1, storekdtree=False)
    ang_dist = d2d.to(u.arcsec).value

    ANGULAR_DISTANCE_MATCH = 3.0
    matched = ang_dist < ANGULAR_DISTANCE_MATCH

    # Keep only the BGS centrals that matched to the Lsat data
    centrals_df = centrals_df.loc[matched].reset_index(drop=True)
    print(f"Of the {orig_bgs_len} BGS centrals, {len(centrals_df)} matched to the Lsat data .")

    centrals_df['Ltot50'] = lsat_df.loc[idx[matched], 'Ltot50'].to_numpy()
    centrals_df['Ltot100'] = lsat_df.loc[idx[matched], 'Ltot100'].to_numpy()
    centrals_df['Rbar'] = lsat_df.loc[idx[matched], 'Rbar'].to_numpy()
    centrals_df['z_from_lsatfile'] = lsat_df.loc[idx[matched], 'z'].to_numpy()
    centrals_df['Dn4k_from_lsatfile'] = lsat_df.loc[idx[matched], 'Dn4k'].to_numpy()
    centrals_df['absmag_r_from_lsatfile'] = lsat_df.loc[idx[matched], 'absmag_r'].to_numpy()

    return centrals_df

In [ ]:
df = get_lsat_for_centrals(LSAT_RAW_2020_OBSERVATIONS_FILE, centrals_df)

In [ ]:
# This is the noisy Lsat measurement that we will measure better by stacking
df['LSAT50'] = df['Ltot50'] - df['Rbar']

In [ ]:
# How many have redshift differences > 0.002?
redshift_diff = np.abs(df['z_from_lsatfile'] - df['Z'])
print(f"Number of galaxies with redshift difference > 0.002: {np.sum(redshift_diff > 0.002)} out of {len(df)} ({np.sum(redshift_diff > 0.002)/len(df)*100:.2f}%)")

# How many have Dn4000 differences > 0.1?
dn4k_diff = np.abs(df['Dn4k_from_lsatfile'] - df['DN4000_MODEL'])
print(f"Number of galaxies with Dn4000 difference > 0.1: {np.sum(dn4k_diff > 0.1)} out of {len(df)} ({np.sum(dn4k_diff > 0.1)/len(df)*100:.2f}%)")

# How many give different quiescent classifications?
df['QUIESCENT_FROM_LSATFILE'] = is_quiescent_SDSS_Dn4000(abs_mag_r_to_log_solar_L(df['absmag_r_from_lsatfile']),  df['Dn4k_from_lsatfile'])
different = df['QUIESCENT_FROM_LSATFILE'] != df['QUIESCENT']
print(f"Number of galaxies with different quiescent classifications: {np.sum(different)} out of {len(df)} ({np.sum(different)/len(df)*100:.2f}%)")

# How many with absmag_r differences > 0.1?
df['ABSMAG_R'] = log_solar_L_to_abs_mag_r(df['LOGLGAL'])
df['L_GAL_from_lsatfile'] = abs_mag_r_to_solar_L(df['absmag_r_from_lsatfile'])
absmag_diff = np.abs(df['absmag_r_from_lsatfile'] - df['ABSMAG_R'])
print(f"Number of galaxies with absmag_r difference > 0.1: {np.sum(absmag_diff > 0.2)} out of {len(df)} ({np.sum(absmag_diff > 0.1)/len(df)*100:.2f}%)")


In [ ]:
# NOTE in C++ code we save of Lsat for a specific bins. Need to update that to match what's here now.
lcen_bins = np.arange(8.7, 11.1, 0.1)

def create_stacked_lsat_calibration_data(df):

    df['STACKING_BIN'] = pd.cut(df['LOGLGAL'], bins=lcen_bins, labels=lcen_bins[:-1])
    # Print how many galaxies in each bin
    #red_gals = df.groupby('STACKING_BIN', observed=True)['QUIESCENT'].sum()
    #blue_gals = df.groupby('STACKING_BIN', observed=True)['QUIESCENT'].size() - red_gals
    #print("Number of red and blue galaxies in each stacking bin:")
    #print(pd.DataFrame({'red': red_gals, 'blue': blue_gals}))

    def get_lsat_for_df(sub_df):
        return sub_df.groupby('STACKING_BIN', observed=True)['LSAT50'].mean()

    lsat_r = get_lsat_for_df(df.loc[df['QUIESCENT']])
    lsat_b = get_lsat_for_df(df.loc[~df['QUIESCENT']])

    # Now bootstrap for error estimation
    N_ITERATIONS = 1000
    def bootstrap_iteration(sample_indices):
        sample = df.iloc[sample_indices]
        lsat_r_samp = get_lsat_for_df(sample.loc[sample['QUIESCENT']])
        lsat_b_samp = get_lsat_for_df(sample.loc[~sample['QUIESCENT']])
        return lsat_r_samp, lsat_b_samp

    # Use Parallel to run bootstraps
    from joblib import Parallel, delayed
    bootstraps = Parallel(n_jobs=-1)(delayed(bootstrap_iteration)(np.random.choice(len(df), size=len(df), replace=True)) for _ in range(N_ITERATIONS))

    lsat_r_bootstrap, lsat_b_bootstrap = zip(*bootstraps)
    lsat_r_err = np.std(lsat_r_bootstrap, axis=0)
    lsat_b_err = np.std(lsat_b_bootstrap, axis=0)
    # Check how std compares to 16-84 percentile range
    #print("Red Lsat error (std vs 16-84 percentile):")
    #print(np.std(lsat_r_bootstrap, axis=0))
    #print((np.percentile(lsat_r_bootstrap, 84, axis=0) - np.percentile(lsat_r_bootstrap, 16, axis=0)) / 2)
    #print("Blue Lsat error (std vs 16-84 percentile):")
    #print(np.std(lsat_b_bootstrap, axis=0))
    #print((np.percentile(lsat_b_bootstrap, 84, axis=0) - np.percentile(lsat_b_bootstrap, 16, axis=0)) / 2)
    # Doesn't matter, it's Gaussian. 

    return lsat_r, lsat_b, lsat_r_err, lsat_b_err
    
lsat_r, lsat_b, lsat_r_err, lsat_b_err = create_stacked_lsat_calibration_data(df)

In [ ]:
# Plot results with error bars, showing y axis as log10(red / blue)
import matplotlib.pyplot as plt
import plotting as pp

old_obs = np.loadtxt(LSAT_OBSERVATIONS_SDSS_TRANSFORMED_FILE, skiprows=0, dtype='float')
obs_lcen = old_obs[:,0] # log10 already
obs_lsat_r = old_obs[:,1]
obs_err_r = old_obs[:,2]
obs_lsat_b = old_obs[:,3]
obs_err_b = old_obs[:,4]
obs_ratio = obs_lsat_r/obs_lsat_b
obs_ratio_err = obs_ratio * ((obs_err_r/obs_lsat_r)**2 + (obs_err_b/obs_lsat_b)**2)**.5
obs_ratio_err_log = obs_ratio_err/obs_ratio/np.log(10)

x_centers = (lcen_bins[:-1] + lcen_bins[1:]) / 2
x = x_centers
y = np.log10(lsat_r / lsat_b)
y_err = np.sqrt((lsat_r_err / lsat_r)**2 + (lsat_b_err / lsat_b)**2) / np.log(10)

fig,axes=plt.subplots(nrows=1, ncols=1, figsize=(5,5), dpi=300)

axes.errorbar(obs_lcen, np.log10(obs_lsat_r/obs_lsat_b), yerr=obs_ratio_err_log, fmt='o', color='k', markersize=3, capsize=3, ecolor='k', label='SDSS Lcen', alpha=0.8)

axes.errorbar(x_centers, y, yerr=y_err, fmt='o', color='green', markersize=3, capsize=3, ecolor='green', label='BGS Lcen)', alpha=0.8)
axes.set_ylabel('log$(L_{\\rm sat}^{q}/L_{\\rm sat}^{sf})$')
axes.set_xlabel('log$(L_{\\rm cen}~/~[L_\odot / h^2])$')
axes.set_ylim(-0.2, 0.5)
axes.set_xticks([9.0, 9.5, 10.0, 10.5, 11.0])
axes.legend()

# Twin x for Mr
ax2=axes.twiny()
ax2.plot(x_centers, y, ls="")
ax2.set_xlim(log_solar_L_to_abs_mag_r(x_centers[0]), log_solar_L_to_abs_mag_r(x_centers[-1]))
ax2.set_xlabel("$M_r$ - 5log($h$)")
ax2.set_xticks([-23, -22, -21, -20, -19, -18, -17])

fig.tight_layout()

In [ ]:
###########################################
# Save off the data to parameters folder
###########################################

# Format Lcen_left, Lsat_r, Lsar_err_r, Lsat_b, Lsat_err_b
calibration_data = np.column_stack([lcen_bins[:-1], lsat_r, lsat_r_err, lsat_b, lsat_b_err])
# Only write 6 significant figures
np.savetxt(PARAMS_BGSY1_FOLDER + LSAT_OBSERVATIONS_FILE, calibration_data, fmt='%.6g')